# **Importing Library**

In [2]:
import pandas as pd
#math operations
import numpy as np
#machine learning
import cv2
import os
from random import shuffle
from tqdm import tqdm
import random
#for opening and loading image
from PIL import Image
#for preprocessing
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
#Doing One hot encoding as classifier has multiple classes
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Dense,Flatten,Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from random import shuffle
#For augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
#MobileNetV2 model
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras import Model, layers
from numpy import loadtxt

import itertools
from sklearn.metrics import confusion_matrix,classification_report

from tensorflow.keras.applications.imagenet_utils import preprocess_input, decode_predictions
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

ModuleNotFoundError: No module named 'tensorflow'

# **Importing Train Dataset**

In [ ]:
import numpy as np

feats_train = np.load('data/feats_train.npy')
labels_train = np.load('data/labels_train.npy')


FileNotFoundError: [Errno 2] No such file or directory: 'data/feats_train.npy'

In [ ]:
# Check shapes of the test features and labels
print(feats_train.shape)
print(labels_train.shape)

(7049, 224, 224, 3)
(7049,)


In [ ]:
num_classes=len(np.unique(labels_train))
len_data=len(feats_train)
print(len_data)
print(num_classes)

7049
4


# **One Hot Encoding**

In [ ]:
from tensorflow.keras.utils import to_categorical

# Assuming labels_train are integer-encoded
num_classes = 5  # Number of classes in your dataset
labels_train_one_hot = to_categorical(labels_train, num_classes=num_classes)


# **Train Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(feats_train, labels_train_one_hot, test_size=0.2, random_state=42)


# **Prepare the Data Generators**

In [ ]:
import tensorflow as tf

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))

# Shuffle and batch the datasets
batch_size = 32

train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)


# **Load the Pre-trained MobileNetV2 Model and Modify It**

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


# **Compile the Model**

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# **Train the Model**

In [ ]:
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    steps_per_epoch=len(X_train) // batch_size,
    validation_steps=len(X_val) // batch_size
)


Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 30s 147ms/step - accuracy: 0.8176 - loss: 0.4824 - val_accuracy: 0.8700 - val_loss: 0.3338
Epoch 2/10
  1/176 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 1.0000 - loss: 0.0506

/Users/shreyasbk/skin_cancer_project/venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 1.0000 - loss: 0.0506 - val_accuracy: 0.8679 - val_loss: 0.3340
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 25s 140ms/step - accuracy: 0.8846 - loss: 0.2876 - val_accuracy: 0.8658 - val_loss: 0.3537
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.8571 - loss: 0.3251 - val_accuracy: 0.8608 - val_loss: 0.3606
Epoch 5/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 25s 141ms/step - accuracy: 0.8949 - loss: 0.2552 - val_accuracy: 0.8899 - val_loss: 0.2999
Epoch 6/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.8571 - loss: 0.4586 - val_accuracy: 0.8871 - val_loss: 0.3134
Epoch 7/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 25s 142ms/step - accuracy: 0.9203 - loss: 0.1978 - val_accuracy: 0.8764 - val_loss: 0.3163
Epoch 8/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.8571 - loss: 0.5674 - val_accuracy: 0.8764 - val_loss: 0.3256
Epoch 9/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 25s 139ms/step - accuracy: 0.9355 - loss: 0.1654 - val_accurac

# **Fine-tune the Model**

In [ ]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    steps_per_epoch=len(X_train) // batch_size,
    validation_steps=len(X_val) // batch_size
)


Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 32s 167ms/step - accuracy: 0.7743 - loss: 0.7291 - val_accuracy: 0.8757 - val_loss: 0.3784
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.7143 - loss: 0.4241 - val_accuracy: 0.8757 - val_loss: 0.3789
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 29s 165ms/step - accuracy: 0.8791 - loss: 0.3156 - val_accuracy: 0.8736 - val_loss: 0.4022
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.8571 - loss: 0.3331 - val_accuracy: 0.8750 - val_loss: 0.4021
Epoch 5/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 29s 165ms/step - accuracy: 0.9027 - loss: 0.2526 - val_accuracy: 0.8771 - val_loss: 0.3881
Epoch 6/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.8571 - loss: 0.4797 - val_accuracy: 0.8771 - val_loss: 0.3874
Epoch 7/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 29s 167ms/step - accuracy: 0.9158 - loss: 0.2216 - val_accuracy: 0.8821 - val_loss: 0.3609
Epoch 8/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.5714 - loss: 0.9614 - 

# Creating the folder to save the model

In [ ]:
import os

# Define the name of the directory you want to create
new_directory_name = "models"

# Define the path of the new directory within the Kaggle output directory
new_directory_path = new_directory_name

# Create the new directory
os.makedirs(new_directory_path)

# Print a message indicating that the directory has been created
print("New directory created:", new_directory_path)


New directory created: models


In [ ]:
# Save the model in HDF5 format.
model.save(f'{new_directory_path}/V2model.h5')


# **Downloading the Model**

# **Importing Test Dataset**

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Load the test data
feats_test = np.load('data/feats_test.npy')
labels_test = np.load('data/labels_test.npy')

# Load your model (assuming it has been previously saved)
loaded_model = tf.keras.models.load_model('models/V2model.h5')


In [ ]:
# Check shapes of the test features and labels
print(feats_test.shape)
print(labels_test.shape)


(1811, 224, 224, 3)
(1811,)


In [ ]:
# Make predictions
predictions = loaded_model.predict(feats_test)
predicted_classes = np.argmax(predictions, axis=1)

# Convert one-hot encoded labels back to integer labels
true_classes = labels_test  # Use the original integer labels


NameError: name 'loaded_model' is not defined

# **Plotting confusion matrix**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

# Generate the confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Plot the confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(num_classes), yticklabels=range(num_classes))
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()
    

Matplotlib is building the font cache; this may take a moment.


NameError: name 'true_classes' is not defined

In [ ]:
# pip install tensorflow requests pillow


# **Predicting class name with probabilities for all classes**

In [ ]:
import tensorflow as tf
# loaded_model = tf.keras.models.load_model('/kaggle/working/skin_model/V2model.h5')
loaded_model = tf.keras.models.load_model('models/V2model.h5')


In [ ]:
#Function to Download and Preprocess Image
import requests
from PIL import Image
from io import BytesIO
import numpy as np

def download_and_preprocess_image(url, target_size=(224, 224)):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    img = img.resize(target_size)
    img_array = np.array(img)

    # Check if the image has an alpha channel (4th channel), and if so, remove it
    if img_array.shape[-1] == 4:
        img_array = img_array[..., :3]

    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)  # Preprocess the image
    return img_array


In [ ]:
# Define Class Names
class_names = ['Basal_cell_carcinoma', 'Melanoma', 'Nevus', 'benign_keratosis', 'no_cancer']  # Update with your actual class names


In [ ]:
#Function to Predict Class with Probabilities
def predict_image_class(url):
    img_array = download_and_preprocess_image(url)
    predictions = loaded_model.predict(img_array)

    predicted_class_index = np.argmax(predictions, axis=1)[0]
    predicted_class_name = class_names[predicted_class_index]

    # Convert predictions to percentages
    prediction_percentages = predictions[0] * 100

    # Print out the prediction percentages for each class
    for i, class_name in enumerate(class_names):
        print(f"{class_name}: {prediction_percentages[i]:.2f}%")

    return predicted_class_name, prediction_percentages



# **Usage**

In [ ]:
# Example usage
image_url = input("Please enter the URL of the image: ")
predicted_class_name, prediction_percentages = predict_image_class(image_url)
print(f'Predicted class: {predicted_class_name}')

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x1310cb740>